In [16]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Added to sys.path:/", repo_root)
from fixedincomelib import *
print("Fixed Income Library is loaded.")

Added to sys.path:/ /Users/nirvana/Documents/NYUQuantClub1
Fixed Income Library is loaded.


### Create Build Method Collection

In [17]:
bm_list = []
# create yield curve build method
content_sofr = {
    'TARGET' : 'SOFR-1B',
    'OVERNIGHT INDEX FUTURE' : 'SOFR-FUTURE-3M'}
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_INDEX', content_sofr))
# create yield curve funding build method
content_funding = {
    'TARGET' : 'SOFR-1B-FLAT',
    'SPREAD ZERO RATE' : 'SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD'}
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_FUNDING', content_funding))
# create yield curve common build method 
content = {
    'TARGET' : 'USD',
    'FUNDING PARAMETERS' : 'USD-FUNDING-PARAMETERS',
    'SOLVER' : 'BRENTQ'}
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_COMMON', content))
build_method_collection = qfCreateModelBuildMethodCollection(bm_list)

### Create Data Collection

In [18]:
### ois futures
df_fut = pd.DataFrame([    
    ['2026-03-19x2026-06-18', 96.44],
    ['2026-06-18x2026-09-17', 96.70],
    ['2026-09-17x2026-12-10', 96.85],
    ['2026-12-10x2027-03-17', 96.90]],
    columns=['axis1', 'values']).set_index('axis1')
data_fut = qfCreateData1D('OVERNIGHT INDEX FUTURE', 'SOFR-FUTURE-3M', df_fut)
### spread zero rate
df_spread_zero_rate = pd.DataFrame([['1Y', 0.]], columns=['axis1', 'values']).set_index('axis1')
data_szr = qfCreateData1D('SPREAD ZERO RATE', 'SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD', df_spread_zero_rate)
### fpt
df_dg = pd.DataFrame([['Overnight Index Future', 'SOFR-FUTURE-3M', 'SOFR-1B-FLAT']], columns=['DATA TYPE', 'DATA CONVENTION', 'FUNDING IDENTIFIER'])
data_fpt = qfCreateDataGeneric('DATA GENERIC', 'USD-FUNDING-PARAMETERS', df_dg)
### pack up all data into data collection
data_collection = qfCreateDataCollection([data_szr, data_fut, data_fpt])

### Create Yield Curve and Val Param

In [19]:
value_date = '2026-02-11'
yc_model = qfCreateModel(value_date, 'YIELD_CURVE', data_collection, build_method_collection)
fi_vp = qfCreateValuationParameters('FUNDING INDEX PARAMETER', {'Funding Index' : 'SOFR-1B-FLAT'})
vp_collection = qfCreateValuationParametersCollection([fi_vp])

### Build Product RFR Future (Calibration Instrument)

In [20]:
effecitve_date = '2026-09-17'
termination_date = '2026-12-10'
future_convention = 'SOFR-FUTURE-3M'
long_or_short = 'LONG'
amount = 1
strike = 96.85
prod_rfr_future_calib = qfCreateProductRFRFuture(
    effecitve_date,
    termination_date,
    future_convention,
    long_or_short,
    amount,
    strike)
qfDisplayProduct(prod_rfr_future_calib)

,Name,Value
0,Product Type,PRODUCT_RFR_FUTURE
1,Notional,250000.0
2,Currency,USD
3,Long Or Short,LONG
4,Effective Date,2026-09-17
5,Termination Date,2026-12-10
6,Future Convention,SOFR-FUTURE-3M
7,Amount,1
8,Strike,96.85


### Test Valuation

In [21]:
### test pv and cash
report = qfCreateValueReport(yc_model, prod_rfr_future_calib, vp_collection, 'pvdetailed')
pv_base = qfCreateValueReport(yc_model, prod_rfr_future_calib, vp_collection, 'pv')[0][1]
display(report.display())

,Currency,Type,Value
0,USD,PV,2.842171e-08
1,USD,CASH,0.000000e+00


In [22]:
### test par rate
par_rate = qfCreateValueReport(yc_model, prod_rfr_future_calib, vp_collection, 'parrateorspread')
print(f'The implied future rate is: {par_rate:.2%}.')

The implied future rate is: 3.15%.


In [23]:
### test pv01
pv01 = qfCreateValueReport(yc_model, prod_rfr_future_calib, vp_collection, 'pv01')
print(f'The pv01 of the future product is: {pv01:.2f}.')

The pv01 of the future product is: 250000.00.


In [24]:
### test cf report
cf_report = qfCreateValueReport(yc_model, prod_rfr_future_calib, vp_collection, 'cashflowsreport')
cf_report.display().T

,0
PRODUCT_TYPE,PRODUCT_RFR_FUTURE
VALUATION_ENGINE_TYPE,ValuationEngineProductRfrFuture
LEG_ID,0
CASHFLOW_ID,0
PAY_OR_RECEIVE,1.0
NOTIONAL,250000.0
PAY_DATE,"September 17th, 2026"
FORECASTED_AMOUNT,0.0
PV,0.0
DISCOUNG FACTOR,1.0


In [25]:
### test risk
risk = qfCreateValueReport(yc_model, prod_rfr_future_calib, vp_collection, 'firstorderrisk')
df_risk = risk.display()
df_risk.VALUES = df_risk.VALUES.round(2)
display(df_risk)

,DATA_TYPE,DATA_CONVENTION,AXIS1,AXIS2,MARKET_QUOTE,UNIT,VALUES
0,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,1Y,,0.0,0.0001,0.0
1,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-03-19x2026-06-18,,96.44,-0.01,0.0
2,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-06-18x2026-09-17,,96.7,-0.01,0.0
3,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-09-17x2026-12-10,,96.85,-0.01,-2500.0
4,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-12-10x2027-03-17,,96.9,-0.01,-0.0


In [26]:
### What is a risk report ?
### Each row shall corresponds to dV/dS_i, where S_i is the MARKET_QUOTE of curve calibration instrument
### It is more readable if we can re-scale according to market convention, i.e., valuation change due to 1 bp move in rates,
### Therefore, we have two columns
### - VALUES: dV/dS_i * UNIT
### - UNIT : most of time, it is 1bp = 0.0001
### In this case, when we need to estimate pnl, it should be
### PnL = change in MARKET_QUOTE * VALUES / UNIT 
###     = \Delta S_i * dV/dS_i

### However, future is special, the MARKET_QUOTE is in price term, so
### Suppose future price is F, the implied rate is S = (100. - F) / 100.
### - VALUES: for consistency, we report dV/dS * 1e-4, i.e., the change of valuation per 1bp change of future implied rate
### - UNIT : we make it -0.01
### Let us look at our PnL prediction formula, does it still make sense ?
### PnL = change in MARKET_QUOTE * VALUES / UNIT
###     =  change in F / -0.01 * (dVdS * 1e-4)
### The first term is the change in rate (per bp) corresponding to the change in F, the second term is change of V due to 1bp change of S
### Everything is consistent !

In [27]:
### Risk Validation
### Two way to validate our risk report:
### 1) Bump & Reval
###    - Suppose base valuation is V, i.e., V = V(M(S))
###    - Rebuild model with pertrubed market instrument M(S + \epsilon)
###    - Value our product with V(epsilon) = V(M(S + \epsilon), Prod)
###    - Risk w.r.t S  = (V(epsilon) - V) / epsilon
###    This gives you a partial w.r.t a single instrument, and you have to loop through all S, which gives you a gradient vector (our risk report)
### 2) PnL Decomposition (in its first order)
###    Suppose market moves S by x, then if our sensitivity calculation is correct in the risk report, we shall have
###    V(M(S + x)) - V(M(S)) \approx dV/dS * x

In [28]:
# Step 1: build bumped up model
bump_size = 1e-4
df_fut_bumped = df_fut.copy()
cur_fut = df_fut_bumped.loc[f'{effecitve_date}x{termination_date}'].values[0]
cur_rate = (100. - cur_fut) / 100.
cur_rate_bumped = cur_rate + bump_size
df_fut_bumped.loc[f'{effecitve_date}x{termination_date}'] = 100 - 100 * cur_rate_bumped
data_fut_bumped = qfCreateData1D('OVERNIGHT INDEX FUTURE', 'SOFR-FUTURE-3M', df_fut_bumped)
data_collection_bumped = qfCreateDataCollection([data_szr, data_fut_bumped, data_fpt])
yc_model_bumped = qfCreateModel(value_date, 'YIELD_CURVE', data_collection_bumped, build_method_collection)
# Step 2: re-value the same product
pv_bumped = qfCreateValueReport(yc_model_bumped, prod_rfr_future_calib, vp_collection, 'pv')[0][1]
# Step 3 : change of value due to change of rate by 1 bp
risk_bump_reval = pv_bumped - pv_base
print(f'Bump reval risk is {risk_bump_reval}.')
# Step 4: bump reval risk - analytic risk
analytic_risk = df_risk[df_risk.AXIS1 == f'{effecitve_date}x{termination_date}']['VALUES'].values[0]
print(f'Analytic risk is {analytic_risk}')

Bump reval risk is -2500.0000000154896.
Analytic risk is -2500.0


In [29]:
### if price moves $1, what is the PnL impact ?
unit = - 0.01
price_move = 1.
rate_move = price_move / unit
pnl_impact = rate_move * analytic_risk
pnl_impact

250000.0

In [30]:
### PnL validation
# build model
df_fut_bumped_price = df_fut.copy()
cur_fut = df_fut_bumped_price.loc[f'{effecitve_date}x{termination_date}'].values[0]
df_fut_bumped_price.loc[f'{effecitve_date}x{termination_date}'] = cur_fut + 1.
data_fut_bumped_price = qfCreateData1D('OVERNIGHT INDEX FUTURE', 'SOFR-FUTURE-3M', df_fut_bumped_price)
data_collection_bumped_price = qfCreateDataCollection([data_szr, data_fut_bumped_price, data_fpt])
yc_model_bumped_price = qfCreateModel(value_date, 'YIELD_CURVE', data_collection_bumped_price, build_method_collection)
# calc value
pv_bumped_price = qfCreateValueReport(yc_model_bumped_price, prod_rfr_future_calib, vp_collection, 'pv')[0][1]
# pnl
pnl_validation = pv_bumped_price - pv_base
print(f'Actual PnL impact is {pnl_validation}.')

Actual PnL impact is 249999.99999997157.


### Build Product RFR Future (Not Calibration Instrument)

In [31]:
effecitve_date = '2026-10-17'
termination_date = '2027-01-17'
future_convention = 'SOFR-FUTURE-3M'
long_or_short = 'SHORT'
amount = 1
strike = 96
prod_rfr_future_no_calib = qfCreateProductRFRFuture(
    effecitve_date,
    termination_date,
    future_convention,
    long_or_short,
    amount,
    strike)
qfDisplayProduct(prod_rfr_future_no_calib)

,Name,Value
0,Product Type,PRODUCT_RFR_FUTURE
1,Notional,250000.0
2,Currency,USD
3,Long Or Short,SHORT
4,Effective Date,2026-10-17
5,Termination Date,2027-01-17
6,Future Convention,SOFR-FUTURE-3M
7,Amount,1
8,Strike,96


In [32]:
### test pv and cash
pv_base = qfCreateValueReport(yc_model, prod_rfr_future_no_calib, vp_collection, 'pv')[0][1]
print(f'base pv is {pv_base:.2f}.')

base pv is -217851.88.


In [33]:
### test risk
df_risk_no_calib = qfCreateValueReport(yc_model, prod_rfr_future_no_calib, vp_collection, 'firstorderrisk').display()
df_risk_no_calib.VALUES = df_risk_no_calib.VALUES.round(2)
df_risk_no_calib

,DATA_TYPE,DATA_CONVENTION,AXIS1,AXIS2,MARKET_QUOTE,UNIT,VALUES
0,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,1Y,,0.0,0.0001,0.00
1,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-03-19x2026-06-18,,96.44,-0.01,0.00
2,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-06-18x2026-09-17,,96.7,-0.01,0.00
3,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-09-17x2026-12-10,,96.85,-0.01,1413.95
4,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-12-10x2027-03-17,,96.9,-0.01,1086.57


In [34]:
### validate risk
risk_tenor = '2026-09-17x2026-12-10'
# Step 1: build bumped up model
bump_size = 1e-4
df_fut_bumped = df_fut.copy()
cur_fut = df_fut_bumped.loc[risk_tenor].values[0]
cur_rate = (100. - cur_fut) / 100.
cur_rate_bumped = cur_rate + bump_size
df_fut_bumped.loc[risk_tenor] = 100 - 100 * cur_rate_bumped
data_fut_bumped = qfCreateData1D('OVERNIGHT INDEX FUTURE', 'SOFR-FUTURE-3M', df_fut_bumped)
data_collection_bumped = qfCreateDataCollection([data_szr, data_fut_bumped, data_fpt])
yc_model_bumped = qfCreateModel(value_date, 'YIELD_CURVE', data_collection_bumped, build_method_collection)
# Step 2: re-value the same product
pv_bumped = qfCreateValueReport(yc_model_bumped, prod_rfr_future_no_calib, vp_collection, 'pv')[0][1]
# Step 3 : change of value due to change of rate by 1 bp
risk_bump_reval = pv_bumped - pv_base
print(f'Bump reval risk is {risk_bump_reval:.2f}.')
# Step 4: bump reval risk - analytic risk
analytic_risk = df_risk_no_calib[df_risk_no_calib.AXIS1 == risk_tenor]['VALUES'].values[0]
print(f'Analytic risk is {analytic_risk}')

Bump reval risk is 1413.94.
Analytic risk is 1413.95


In [35]:
### validate risk
risk_tenor = '2026-12-10x2027-03-17'
# Step 1: build bumped up model
bump_size = 1e-4
df_fut_bumped = df_fut.copy()
cur_fut = df_fut_bumped.loc[risk_tenor].values[0]
cur_rate = (100. - cur_fut) / 100.
cur_rate_bumped = cur_rate + bump_size
df_fut_bumped.loc[risk_tenor] = 100 - 100 * cur_rate_bumped
data_fut_bumped = qfCreateData1D('OVERNIGHT INDEX FUTURE', 'SOFR-FUTURE-3M', df_fut_bumped)
data_collection_bumped = qfCreateDataCollection([data_szr, data_fut_bumped, data_fpt])
yc_model_bumped = qfCreateModel(value_date, 'YIELD_CURVE', data_collection_bumped, build_method_collection)
# Step 2: re-value the same product
pv_bumped = qfCreateValueReport(yc_model_bumped, prod_rfr_future_no_calib, vp_collection, 'pv')[0][1]
# Step 3 : change of value due to change of rate by 1 bp
risk_bump_reval = pv_bumped - pv_base
print(f'Bump reval risk is {risk_bump_reval:.2f}.')
# Step 4: bump reval risk - analytic risk
analytic_risk = df_risk_no_calib[df_risk_no_calib.AXIS1 == risk_tenor]['VALUES'].values[0]
print(f'Analytic risk is {analytic_risk}')

Bump reval risk is 1086.56.
Analytic risk is 1086.57
